# Phase 1: routing data collection (Colab)

Collects Phase 1 routing telemetry from Qwen1.5-MoE-A2.7B (fp16) and validates it.

**What it does:**
1. Checks for a GPU, clones the repo at the chosen branch or commit, installs the pinned requirements, and confirms the installed versions match.
2. Collects **every prompt once** (GSM8K train + test, all MBPP splits: about 9,800 prompts) into `full/`, plus a stratified **200-prompt subset per dataset in 5 orderings** into `determinism/` (#109).
3. Reruns ordering 0 of the subset, so the same-batch check can compare identical batches (#134).
4. Validates the data with the determinism check (`validation_report.json`). The collection writes `manifest.json` next to the data: the model, settings, software versions, GPU, commit and results (#134).
5. Copies everything to a Drive folder named after the commit.

**Runtime:** about 6 minutes of collection on an A100 (measured at `38ef1da`, before the same-batch repeat added 400 prompt-passes), plus a few minutes of setup and validation, plus the copy to Drive (about 3.3 GB).

**Before you start:** Runtime → Change runtime type → **A100 GPU**. Colab may give you the 40 GB or the 80 GB A100; `manifest.json` records which.

**When you're done:** disconnect the runtime (last section). Colab bills for GPU time as long as it's connected.

## Settings

In [ ]:
# Branch, tag or commit to collect from. The official collection (#135) uses "main";
# set a branch or commit for a test run.
REF = "main"

NUM_ORDERINGS = 5      # orderings of the determinism subset (#109)
SUBSET_SIZE = None     # None = every prompt. A number limits prompts per dataset, for a quick test run.
COPY_TO_DRIVE = True   # results are written to local disk first, then copied here:
DRIVE_FOLDER = "/content/drive/My Drive/moe_phase1_data"
# The model's near-tie window record, from the repository (#128). The validator checks
# it is for the model the collection ran (from manifest.json), and stops if not (#134).
WINDOW_RECORD = "phase_1/near_tie_windows/qwen1.5-moe-a2.7b.json"


## Setup

In [ ]:
# Fail fast if there's no GPU, before cloning or installing anything
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU. Runtime → Change runtime type → A100 GPU, then run again."
print("GPU:", gpu.stdout.strip())

In [ ]:
# Mount Drive now, so the sign-in prompt doesn't block an unattended run at the end
if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Clone the repo at REF and install the pinned requirements
REPO = "/content/moe-expert-prefetching"
!rm -rf {REPO}
!git clone --quiet https://github.com/scerruti/moe-expert-prefetching.git {REPO}
!git -C {REPO} checkout --quiet {REF}
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Collecting from {REF} at commit {COMMIT}")

!pip install -q -r {REPO}/requirements.txt

In [ ]:
# Confirm the installed packages match requirements.txt (one test per pin)
!cd {REPO} && python -m pytest phase_1/scripts/test_pinned_versions.py -q -p no:cacheprovider

## Run the collection

Writes to local disk (`/content/phase1_<commit>`): much faster than writing through the Drive mount. Progress and a per-stage timing breakdown are logged as it runs.

In [ ]:
import json
import logging
import sys
import time
from pathlib import Path

# Python reuses modules it has already imported, so a second run in the same
# session would silently use the previous REF's code
stale = [m for m in ("run_settings", "full_pipeline", "forward_pass", "pytorch_forward_hooks",
                     "determinism_validation")
         if m in sys.modules]
assert not stale, (f"Code from an earlier run is still loaded ({', '.join(stale)}). "
                   "Runtime → Restart session, then Run all again.")

sys.path.insert(0, f"{REPO}/phase_1/scripts")
sys.path.insert(0, REPO)  # run_settings.py is at the repository root

# Before torch: Triton reads TRITON_DEFAULT_FP_FUSION when it starts (#133).
# Stops if the environment sets it differently, or if torch is already imported.
from run_settings import apply_run_settings
print("Run settings:", apply_run_settings())
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)

import torch
from full_pipeline import run_full_pipeline

output_dir = Path(f"/content/phase1_{COMMIT}")
assert not output_dir.exists() or not any(output_dir.iterdir()), (
    f"{output_dir} already has data. Delete it, or collect from a different REF.")
output_dir.mkdir(parents=True, exist_ok=True)

# The collection writes manifest.json itself, before any data, and adds the results at the end (#134)
start = time.time()
stats = run_full_pipeline(num_runs=NUM_ORDERINGS, subset_size=SUBSET_SIZE, output_dir=output_dir, ref=REF)
minutes = (time.time() - start) / 60
manifest = json.loads((output_dir / "manifest.json").read_text())

print(f"\nCollected in {minutes:.1f} min: {stats.total_prompts_processed:,} prompt-passes, "
      f"{stats.total_routing_events:,} rows")
print(f"Full passes {stats.full_passes_completed}/2, orderings {stats.runs_completed}/{NUM_ORDERINGS}, "
      f"same-batch repeat {stats.repeats_completed}/1, errors: {stats.errors or 'none'}")
print(f"Manifest: status {manifest['status']}, commit {manifest['code']['commit'][:7]}"
      f"{' (with uncommitted changes)' if manifest['code']['uncommitted_changes'] else ''}, "
      f"GPU {(manifest['gpu'] or {}).get('name')}")


## Validate

The determinism check decided in `phase_1/docs/determinism_check.md` (#128):

- **Same-batch:** ordering 0 and its repeat, identical batches, must give bitwise-identical routing.
- **Near-tie:** routing changes slightly with which prompts share a batch (#114). Each root change between orderings is compared with the model's near-tie window. Root changes outside it are listed as **warnings to investigate**, not failures.

The status is **passed**, **incomplete** (a check couldn't run) or **failed**. The validator stops before checking anything if `manifest.json` is missing, the run didn't finish, or the window record is for a different model (#134).


In [ ]:
from determinism_validation import DeterminismValidator

validator = DeterminismValidator(output_dir, window=Path(REPO) / WINDOW_RECORD)
report = validator.validate_all()
validator.save_report(report, output_dir / "validation_report.json")
print(report.summary)


In [ ]:
# Everything written, with sizes
total = 0
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        total += f.stat().st_size
        print(f"  {str(f.relative_to(output_dir)):40s} {f.stat().st_size / 1e6:8.1f} MB")
print(f"  {'total':40s} {total / 1e6:8.1f} MB")

## Copy to Drive

Copies the whole output folder to a Drive subfolder named after the commit, for example `moe_phase1_data/phase1_38ef1da/`. About 3.3 GB for a full collection, so it takes a few minutes and uses Drive quota.

In [ ]:
if COPY_TO_DRIVE:
    target = Path(DRIVE_FOLDER) / output_dir.name
    assert not target.exists(), f"{target} already exists on Drive; not overwriting it"
    target.parent.mkdir(parents=True, exist_ok=True)
    !cp -r "{output_dir}" "{target}"
    !du -sh "{target}"
    print(f"Copied to {target}")
else:
    print(f"Not copied. The results are only on this runtime's local disk: {output_dir}")

## Finish

1. Check `manifest.json` and `validation_report.json` were copied, and that the manifest's status is `finished`.
2. **Runtime → Disconnect and delete runtime**, to stop GPU billing. This wipes the local disk, so copy first.
3. Record the run on the relevant issue: commit, GPU, time, and the validation summary.

**To run something else on this GPU first** (another `REF`, or GPU tests in the terminal): **Runtime → Restart session**. This notebook's kernel keeps the model (about 30 GB) in GPU memory, and keeps the previous run's code loaded.